# Amazon Data Engineering Interview Questions

**CTC:** 25 LPA | **Experience:** 4+ Years

## 🔹 Round 1 – Technical

**1.** Explain how you created a data model during experimentation and A/B testing.

**2.** How would you find the **Nth highest salary** within each department using SQL? Why would you choose **DENSE_RANK** over **RANK**?

**3.** How do you optimize **Spark jobs** that are taking longer than expected?

**4.** How would you upload **Parquet files to Azure Data Lake Storage** using Python and Azure SDK?

**5.** How does **Airflow operate in a Kubernetes environment**, and how does it store logs?

**6.** Design an **event-driven architecture** for an analytics platform like Mixpanel.

**7.** How would you perform an **upsert in Delta Lake** stored in Azure Data Lake using Spark?

**8.** What is the difference between **repartition() and coalesce()** in Spark, and when would you use each?

---





### 1. Explain how you created a data model during experimentation and A/B testing.

The interviewer is basically asking:

##### “How did you organize and structure data so that the business could compare two versions of a product or feature and determine which one performed better?”

“In my project, I created a data model to analyze A/B testing results by combining user, experiment assignment, and event data.

1. Understand business requirements: Identify the experiment objective, such as measuring whether a new feature improves conversion rate or user engagement.
2. Identify data sources: Collect user information, experiment group assignments (control and treatment), and event data such as clicks, purchases, and sessions.
3. Design the data model: Create fact and dimension tables.
   - `dim_users`: User details and attributes.
   - `dim_experiments`: Experiment ID, name, start date, and end date.
   - `fact_experiment_assignments`: User ID, experiment ID, and assigned group.
   - `fact_user_events`: User ID, event type, timestamp, and transaction amount.
4. Build ETL pipelines: Use Azure Data Factory and Databricks to ingest, clean, deduplicate, and transform the data into Delta tables.
5. Define metrics: Calculate conversion rate, average order value, click-through rate, and user engagement for each experiment group.
6. Ensure data quality: Validate user assignments, remove duplicate events, handle late-arriving data, and ensure users are consistently assigned to the same experiment group.
7. Support analysis: Aggregate metrics by experiment group and provide the resulting datasets to analysts or BI dashboards for statistical significance testing.




![image_1792852701147.png](./image_1792852701147.png "image_1792852701147.png")



### Example SQL query

Calculate the conversion rate for each A/B testing group:

```
SELECT
    a.experiment_id,
    a.variant,
    COUNT(DISTINCT a.user_id) AS total_users,
    COUNT(DISTINCT CASE
        WHEN e.event_type = 'purchase'
        THEN a.user_id
    END) AS converted_users,
    1.0 * COUNT(DISTINCT CASE
        WHEN e.event_type = 'purchase'
        THEN a.user_id
    END) / NULLIF(COUNT(DISTINCT a.user_id), 0)
        AS conversion_rate
FROM fact_experiment_assignments a
LEFT JOIN fact_user_events e
    ON a.user_id = e.user_id
GROUP BY
    a.experiment_id,
    a.variant;
```

Important interview point: In a production implementation, join events to assignments using `experiment_id` and the experiment's eligibility period as well as `user_id`, to avoid attributing unrelated events to an experiment.

If the interviewer asks about your role: Emphasize that, as a Data Engineer, your responsibility was to build reliable, scalable, and accurate datasets for experimentation. Statistical hypothesis testing and deciding whether a result is significant would typically be handled by analysts or data scientists, in collaboration with the engineering team.

### 2. SQL Query: Nth Highest Salary in Each Department

Use `DENSE_RANK()` to rank distinct salary values within each department.

SQL

```
WITH ranked_employees AS (
    SELECT
        employee_id,
        employee_name,
        department_id,
        salary,
        DENSE_RANK() OVER (
            PARTITION BY department_id
            ORDER BY salary DESC
        ) AS salary_rank
    FROM employees
)
SELECT
    employee_id,
    employee_name,
    department_id,
    salary
FROM ranked_employees
WHERE salary_rank = N;
```

How it works:

1. `PARTITION BY department_id` ranks employees separately within each department.

2. `ORDER BY salary DESC` ranks the highest salary first.

3. `DENSE_RANK()` assigns the same rank to employees with equal salaries, without skipping subsequent ranks.

4. `WHERE salary_rank = N` returns employees earning the Nth highest distinct salary.

Note: Replace `N` with the required rank, such as `3` for the third-highest salary.

### 2. Why choose `DENSE_RANK()` over `RANK()`?

Consider these salaries in descending order:

| Salary  | `RANK()` | `DENSE_RANK()` |
| ------- | -------- | -------------- |
| 120,000 | 1        | 1              |
| 100,000 | 2        | 2              |
| 100,000 | 2        | 2              |
| 80,000  | 4        | 3              |
| 70,000  | 5        | 4              |

* `RANK()` skips rank numbers after ties.

* `DENSE_RANK()` does not skip rank numbers after ties.

For example, to find the 3rd highest distinct salary:

* `RANK() = 3` returns no rows in this example.

* `DENSE_RANK() = 3` returns the employee(s) earning `80,000`.

###  Interview-ready answer

“I use `DENSE_RANK()` with `PARTITION BY department_id` and `ORDER BY salary DESC` to find the Nth highest salary within each department. I prefer `DENSE_RANK()` because it ranks distinct salary values consecutively, even when multiple employees have the same salary. `RANK()` skips subsequent rank numbers after ties, which can cause the Nth rank to be missing. Therefore, `DENSE_RANK()` is ideal when the requirement is the Nth highest distinct salary.”

Remember: Use `ROW_NUMBER()` if you need exactly the Nth employee in a deterministic ordering, `RANK()` if gaps after ties are intended, and `DENSE_RANK()` if you need the Nth distinct salary.


### 3. How do you optimize Spark jobs that are taking longer than expected?

“I follow a systematic approach to identify the bottleneck and optimize the Spark job without unnecessarily increasing the cost.”

1. Analyze Spark UI: Check job, stage, and task execution times to identify slow stages, shuffle operations, and long-running tasks.

2. Optimize data partitioning: Use `repartition()` to increase parallelism when needed and `coalesce()` to reduce partitions without a full shuffle.

3. Handle data skew: Identify uneven partition sizes and apply salting, broadcast joins, or Adaptive Query Execution (AQE), depending on the scenario.

4. Optimize joins: Use broadcast joins for small tables and sort-merge joins for large tables. Filter data before joining.

5. Reduce shuffle operations: Avoid unnecessary `groupBy`, `distinct`, and repartitioning. Prefer efficient aggregations where possible.

6. Optimize file formats: Use Parquet or Delta Lake with predicate pushdown, partition pruning, and appropriately sized files.

7. Improve transformations: Avoid Python UDFs when built-in Spark SQL functions can perform the same operation. Select only the required columns and filter early.

8. Use caching wisely: Cache or persist DataFrames reused across multiple actions, and unpersist them when no longer needed.

9. Tune resources: Check executor memory, cores, garbage collection, and task concurrency. Scale resources only when the job is resource-constrained.

10. Re-run and compare: Measure execution time, shuffle read/write, spill, and resource utilization before and after optimization.

### Example: Slow join in PySpark

Python

Run

```
from pyspark.sql.functions import broadcast

# Filter early and select required columns
filtered_orders = orders.filter("status = 'COMPLETED'") \
                        .select("customer_id", "amount")

# Broadcast the smaller dimension table
result = filtered_orders.join(
    broadcast(customers.select("customer_id", "region")),
    on="customer_id",
    how="inner"
)
```

This can reduce data shuffling when `customers` is sufficiently small to fit safely in executor memory.

### Key interview point

I first identify the bottleneck using Spark UI, then optimize the specific issue. For example, if one task is much slower than others, I investigate data skew; if shuffle read/write is high, I optimize joins and partitioning; if tasks spill heavily, I investigate memory pressure and partition sizes.

This approach improves performance while balancing execution time, cluster cost, and reliability.


### 4. Upload Parquet Files to ADLS using Python and Azure SDK

I use the Azure Data Lake Storage Gen2 SDK (`azure-storage-file-datalake`) to upload Parquet files. I typically authenticate using Microsoft Entra ID and `DefaultAzureCredential`, avoiding hardcoded credentials.

### 1. Install required libraries

Bash

```
pip install azure-identity azure-storage-file-datalake
```

### 2. Python code to upload a Parquet file

Python

Run

```
from azure.identity import DefaultAzureCredential
from azure.storage.filedatalake import DataLakeServiceClient

# ADLS Gen2 configuration
account_name = "mystorageaccount"
file_system_name = "raw"
directory_name = "parquet"
local_file_path = "data.parquet"
remote_file_name = "data.parquet"

# Authenticate using Microsoft Entra ID
account_url = f"https://{account_name}.dfs.core.windows.net"
credential = DefaultAzureCredential()

service_client = DataLakeServiceClient(
    account_url=account_url,
    credential=credential
)

# Get container (file system) and directory clients
file_system_client = service_client.get_file_system_client(
    file_system_name
)

directory_client = file_system_client.get_directory_client(
    directory_name
)

# Create directory if it doesn't exist
try:
    directory_client.create_directory()
except Exception:
    pass

# Upload the Parquet file
file_client = directory_client.get_file_client(remote_file_name)

with open(local_file_path, "rb") as file_data:
    file_client.upload_data(file_data, overwrite=True)

print("Parquet file uploaded successfully!")
```

### 3. Key steps to explain in an interview

1. Authentication: Use `DefaultAzureCredential` with a managed identity or service principal.

2. Connection: Create a `DataLakeServiceClient` using the ADLS Gen2 DFS endpoint.

3. Container access: Get the file system and directory clients.

4. Upload: Read the local Parquet file in binary mode and call `upload_data()`.

5. Overwrite handling: Set `overwrite=True` if an existing file should be replaced.

6. Security: Grant the identity appropriate Azure RBAC permissions and ADLS directory/file-system ACLs where required.

Important: The storage account must have hierarchical namespace enabled for ADLS Gen2. Also, avoid silently ignoring all directory-creation exceptions in production; handle “already exists” separately from permission or connectivity errors.

### 4. Practical data engineering tip

If the Parquet data comes from a PySpark DataFrame, you can write it directly to ADLS using Spark's `abfss://` path rather than collecting data locally and uploading it through Python. This is generally more scalable for large datasets.


## 5. How does Airflow operate in Kubernetes, and how does it store logs?

Answer:

“Apache Airflow runs on Kubernetes by deploying its components as containers or pods. Kubernetes manages scheduling, scaling, networking, and recovery, while Airflow orchestrates the workflows.”

### 1. How Airflow operates in Kubernetes

1. Scheduler: Parses DAGs, monitors dependencies, and schedules tasks.

2. Webserver / API server: Provides the UI or API to monitor DAG runs and task status.

3. Metadata database: Stores DAG runs, task states, connections, and other orchestration metadata in PostgreSQL or MySQL.

4. Executor: Determines how tasks run. With the KubernetesExecutor, Airflow creates a separate Kubernetes pod for each task instance.

5. Workers: With CeleryExecutor, tasks run in worker pods instead; those workers can be scaled independently.

6. DAG distribution: DAG files are made available to Airflow components using images, persistent volumes, or synchronized storage.

### 2. How Airflow stores logs

Airflow supports two main logging approaches:

Local logs

Task logs are written to the container's local filesystem. With Kubernetes, these files may disappear when a task pod is deleted unless logs are persisted or collected elsewhere.

Remote logs — recommended for production

Airflow uploads task logs to durable storage such as Amazon S3, Google Cloud Storage, or Azure Blob Storage. Logs remain accessible after worker or task pods terminate.

### 3. Example architecture

Airflow Scheduler

Kubernetes API

Creates task pods

Task Pods

Execute Python, SQL, Spark or ETL tasks

Remote Log Storage

S3 / GCS / Azure Blob Storage

### 4. How to configure remote logging

For example, with an appropriate Airflow provider and permissions, remote logging can be configured in `airflow.cfg`:

INI

```
[logging]
remote_logging = True
remote_base_log_folder = wasb://<container>@<account>.blob.core.windows.net/airflow-logs
remote_log_conn_id = <azure_blob_connection_id>
```

The exact connection type and supported URI format depend on the installed Airflow Azure provider version. Credentials should be supplied through a secure Airflow connection or workload identity, not hardcoded.

### 5. Important production considerations

* Use remote logging so logs survive pod termination.

* Configure Kubernetes resource requests and limits.

* Use a persistent, shared metadata database.

* Apply RBAC and secure credentials.

* Configure log retention and centralized monitoring.

* Distinguish Airflow task logs from Kubernetes container logs, which may separately be collected by Fluent Bit, Azure Monitor, or another logging agent.

Strong interview conclusion: “Kubernetes manages the lifecycle of Airflow task pods, while Airflow manages workflow dependencies and execution state. I prefer remote task logging because Kubernetes pods are ephemeral, and durable centralized logs are essential for debugging failed pipelines and auditing production workloads.”


# 6. Event-Driven Analytics Platform Like Mixpanel

Goal: Design a scalable, real-time analytics platform that ingests millions of user events, processes them reliably, and supports dashboards for funnels, retention, user journeys, and A/B testing.

For a data engineering interview, I would design this using Kafka, Spark Structured Streaming, cloud object storage, a real-time OLAP database, and a batch processing layer.

## 1. High-level architecture

1. Event Producers

Web SDK · Mobile SDK · Backend APIs

2. Event Collection API

Authentication · Validation · Rate limiting · Enrichment

3. Kafka / Event Bus

Partitioned topics · Replication · Buffering · Replay

Real-time path

Spark Structured Streaming

ClickHouse / OLAP store

Fast dashboards and funnels

Historical path

ADLS / S3 + Delta Lake

Batch ETL / Spark SQL

Historical analysis and backfills

4. Query API and Analytics UI

Funnels · Retention · Cohorts · User journeys · Experiments

## 2. Event data model

Every incoming event should follow a standard schema so that different SDKs and applications can send data consistently.

Example event:

JSON

```
{
  "event_id": "evt_10001",
  "tenant_id": "tenant_123",
  "user_id": "user_456",
  "event_name": "purchase_completed",
  "event_time": "2026-10-10T10:30:00Z",
  "ingestion_time": "2026-10-10T10:30:02Z",
  "properties": {
    "product_id": "P101",
    "amount": 2500,
    "currency": "INR"
  },
  "schema_version": 1
}
```

Key design decisions:

* `event_id`: supports deduplication.

* `tenant_id`: separates customer data in a multi-tenant platform.

* `user_id`: supports user-level analytics and retention.

* `event_time`: represents when the event actually occurred.

* `ingestion_time`: helps measure ingestion delays.

* `properties`: stores event-specific attributes.

* `schema_version`: supports backward-compatible schema evolution.

## 3. End-to-end event processing

1. Ingest: SDKs send events to a stateless API gateway. Validate authentication, tenant quotas, payload size, and required fields.

2. Publish: Send valid events to Kafka, partitioned by a suitable key such as `tenant_id` or a combination of tenant and user identifiers. Choose the key based on ordering and load distribution requirements.

3. Stream process: Spark Structured Streaming reads Kafka events, validates schemas, enriches events, deduplicates by `event_id`, and handles late-arriving events with event-time watermarks.

4. Store raw data: Persist immutable raw events in ADLS Gen2 or Amazon S3, partitioned by event date and, where appropriate, tenant. Use Delta Lake for reliable curated datasets.

5. Build analytics tables: Aggregate events into hourly or daily metrics, funnels, retention cohorts, and user activity summaries.

6. Serve queries: Store query-optimized aggregates and event data in an OLAP database such as ClickHouse. A query API applies tenant authorization and returns results to the dashboard.

## 4. How would you scale to millions of events?

I would focus on four areas.

| Component       | Scaling strategy                                                                                      |
| --------------- | ----------------------------------------------------------------------------------------------------- |
| Collection API  | Stateless services with horizontal autoscaling and rate limits                                        |
| Kafka           | Increase partitions and brokers based on throughput, retention, and consumer parallelism              |
| Spark           | Tune partition counts, batch sizes, shuffle operations, and autoscaling                               |
| Analytics store | Partition by time, use appropriate sort keys, pre-aggregate common queries, and scale storage/compute |

Example sizing exercise: Assume 1 million events per second, with an average serialized event size of 1 KB.

* Ingestion throughput: approximately 1 GB/second before protocol overhead.

* Daily raw volume: approximately 86.4 TB/day before compression.

* Kafka capacity, replication overhead, retention, and consumer lag must be included in the capacity plan.

These are planning estimates, not benchmark results. I would validate them using load testing and representative event sizes.

## 5. Reliability and failure handling

This is one of the most important parts of the interview.

* Duplicate events: Use stable event IDs and idempotent writes or deduplication.

* Consumer failure: Resume from committed Kafka offsets and checkpointed streaming state.

* Malformed events: Send invalid records to a dead-letter topic with failure metadata.

* Late events: Use event-time processing and watermarks, with a correction or backfill strategy for events arriving beyond the allowed lateness.

* Schema changes: Enforce schema compatibility rules and version event contracts.

* Backpressure: Monitor consumer lag and scale consumers or adjust processing throughput.

* Data loss: Use Kafka replication, durable raw storage, and carefully coordinated checkpoint/offset recovery.

A key distinction: Kafka delivery guarantees and Spark processing guarantees alone do not automatically provide end-to-end exactly-once behavior across every sink. I would design sink writes to be idempotent and test recovery scenarios.

## 6. Real-time analytics and A/B testing

For example, a product team wants to measure whether a redesigned checkout page improves conversion.

I would:

1. Capture `checkout_started` and `purchase_completed` events.

2. Join events to experiment assignments.

3. Aggregate unique users and conversions by experiment variant.

4. Store the resulting metrics in the OLAP layer.

5. Expose conversion rates and funnel drop-offs through the dashboard.

The experiment assignment must be consistent, and the metric calculation must avoid duplicate users and purchases.

## 7. Monitoring and security

Monitoring

Track events per second, ingestion latency, Kafka consumer lag, Spark processing duration, failed records, data freshness, and query latency.

Security

Enforce tenant-level access controls, encryption in transit and at rest, PII masking, retention policies, audit logging, and deletion workflows.

Cost optimization

Use compressed columnar storage, lifecycle policies, incremental aggregation, autoscaling, and precomputed results for frequent dashboard queries.

## 8. Strong interview conclusion

“I would use Kafka as the durable event backbone, Spark Structured Streaming for scalable real-time processing, ADLS with Delta Lake for historical storage and replay, and ClickHouse or another OLAP engine for low-latency analytics queries. I would design for at-least-once delivery with idempotent processing, schema evolution, event-time handling, and tenant isolation. Finally, I would monitor end-to-end latency, data quality, consumer lag, and cost to ensure the platform remains reliable at scale.”


## 7. How do you perform an UPSERT in Delta Lake using Spark?

Answer:

“I use Delta Lake’s `MERGE INTO` operation to perform an UPSERT, which updates existing records and inserts new records based on a matching key. Delta Lake stores the data in Azure Data Lake Storage Gen2, accessed through an `abfss://` path.”

### 1. PySpark code example

Assume we have a target Delta table containing customer data and a source DataFrame containing new or updated records.

Python

Run

```
from delta.tables import DeltaTable

# Target Delta table in ADLS Gen2
delta_path = (
    "abfss://curated@myaccount.dfs.core.windows.net/"
    "customers"
)

# Source DataFrame containing incoming records
source_df = incoming_df

# Load the existing Delta table
target = DeltaTable.forPath(spark, delta_path)

# Perform UPSERT
(
    target.alias("t")
    .merge(
        source_df.alias("s"),
        "t.customer_id = s.customer_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
```

### 2. How it works

1. Match: Compare source and target records using `customer_id`.

2. Update: If the customer already exists, update its columns.

3. Insert: If the customer does not exist, insert a new record.

4. Atomic commit: Delta Lake commits the transaction atomically, maintaining table consistency.

### 3. What if the target table does not exist?

Create it first:

Python

Run

```
(
    incoming_df.write
    .format("delta")
    .mode("overwrite")
    .save(delta_path)
)
```

Use this initialization only when the target table is absent or when a deliberate full replacement is intended. Do not blindly overwrite an existing production table.

### 4. Important production considerations

* Duplicate source keys: Deduplicate the source before merging; multiple source rows matching the same target row can cause merge errors or ambiguous updates.

* Incremental processing: Read only new or changed source records using a watermark or CDC.

* Performance: Partition or optimize data based on query patterns; avoid over-partitioning and use selective merge conditions when safe.

* Schema evolution: Handle new columns explicitly and enable schema evolution only when appropriate.

* Concurrency: Handle concurrent writes and retry transient transaction conflicts safely.

* Security: Use managed identity or service-principal authentication for ADLS access.

### 5. Interview-ready summary

“I use Delta Lake `MERGE INTO` through the PySpark DeltaTable API. The merge condition identifies matching business keys; matched records are updated, and unmatched records are inserted. Delta Lake provides ACID transactions, while deduplication, incremental ingestion, and merge optimization help ensure reliable and performant data pipelines.”


## 8. Difference between `repartition()` and `coalesce()` in Spark

Both functions change the number of partitions in a Spark DataFrame, but the main difference is shuffling and performance.

### 1. Key differences

| Feature           | `repartition()`                           | `coalesce()`                                   |
| ----------------- | ----------------------------------------- | ---------------------------------------------- |
| Shuffle           | Usually performs a full shuffle           | Avoids a full shuffle when reducing partitions |
| Partitions        | Increases or decreases                    | Primarily reduces                              |
| Data distribution | More balanced distribution                | May produce uneven partitions                  |
| Performance       | More expensive due to shuffle             | Usually faster for reducing partitions         |
| Best use case     | Increase parallelism or redistribute data | Reduce partitions before writing output        |

### 2. Example in PySpark

Using `repartition()`

Python

Run

```
# Increase partitions from 10 to 50
df_repartitioned = df.repartition(50)
```

Use this when you need more parallelism or want to redistribute data across partitions, such as before a large join.

You can also repartition by a key:

Python

Run

```
df_repartitioned = df.repartition(50, "customer_id")
```

Using `coalesce()`

Python

Run

```
# Reduce partitions from 50 to 10
df_coalesced = df.coalesce(10)
```

Use this when a DataFrame has too many partitions and you want to reduce the number of output files without a full shuffle.

### 3. Real-world data engineering scenarios

* Scenario 1 — Large dataset with insufficient parallelism: Use `repartition()` to increase partitions and improve task distribution.

* Scenario 2 — Writing data to ADLS: Use `coalesce()` to reduce the number of output files when the existing partitions are excessive.

* Scenario 3 — Data skew: `repartition()` alone may not fix skew; use techniques such as key salting or Adaptive Query Execution where appropriate.

### 4. Interview-ready answer

“`repartition()` performs a shuffle to redistribute data and can increase or decrease the number of partitions. I use it when I need better parallelism or balanced data distribution. `coalesce()` typically reduces partitions without a full shuffle, so I use it to reduce overhead, especially before writing output files. However, coalesce can create uneven partition sizes, so I choose based on the data distribution and workload.”

Remember: `repartition()` = redistribution; `coalesce()` = efficient partition reduction.
